In [0]:
from pyspark.sql import functions as F

MIN_SALE_AMOUNT = 10000            # sales below this are treated as gifts/family transfers
RES_CLASSES = [
    "Res Condo Common Area",
    "Residential Condominium",
    "Residential Duplex",
    "Residential SFR",
    "Residential Triplex",
    "Townhome",
]

bronze = spark.table("gwinnett.bronze.property_ownership_raw")
latest_q = bronze.agg(F.max("source_quarter")).first()[0]   # the newest quarter loaded

parcels = (bronze
    .filter(F.col("source_quarter") == latest_q)            # only process the newest download
    .filter(F.col("PROPCLAS_DESC") == "R")                  # residential group
    .filter(F.trim(F.col("PCDESC")).isin(RES_CLASSES))      # only the property types you chose
)
print(latest_q, parcels.count())

In [0]:
sales = (parcels
    .selectExpr(
        "PIN AS pin",
        "LRSNum AS lrsnum",
        "PROPERTYSTREET AS street",
        "PROPERTYCity AS city",
        "PROPERTYZip AS zip",
        "Public_NeiNum AS neighborhood",
        "PROPCLAS AS property_class",
        "trim(PCDESC) AS class_desc",
        "try_cast(TOTVAL1 AS DOUBLE) AS appraised_value",
        "try_cast(left(ASSMNT1D, 10) AS DATE) AS assessment_date",
        "source_quarter",
        # stack turns the 3 side-by-side sale pairs into 3 rows per parcel
        "stack(3, SALE1D, SALE1AMT, SALE2D, SALE2AMT, SALE3D, SALE3AMT) AS (sale_date_raw, sale_amount_raw)"
    )
    .withColumn("sale_date", F.expr("try_cast(left(sale_date_raw, 10) AS DATE)"))   # text -> date, bad values become null
    .withColumn("sale_amount", F.expr("try_cast(sale_amount_raw AS DOUBLE)"))       # text -> number, bad values become null
    .drop("sale_date_raw", "sale_amount_raw")
    .filter(F.col("sale_date").isNotNull() & (F.col("sale_date") <= F.current_date()))  # must have a real, non-future date
    .filter(F.col("sale_amount") >= MIN_SALE_AMOUNT)                                    # drops the $0 / $1 transfers
    .dropDuplicates(["pin", "sale_date", "sale_amount"])                                # same sale listed in two slots
)
print(sales.count())